# Get to Know a Dataset: WildNetwork open wildlife detections (WDX)

This notebook is a guided tour of the [WildNetwork open wildlife detections (WDX)](https://registry.opendata.aws/wildnetwork-wdx) dataset on the Registry of Open Data on AWS.

WildNetwork is a free, open live map of wildlife detections from community acoustic stations (BirdNET-Pi, BirdNET-Go, BirdWeather, the open WildNetwork Base), AI camera traps and citizen science (iNaturalist, GBIF). Every record is a [WDX](https://github.com/arunrajiah/wildlife-detection-exchange) event that keeps the species, time and place together with the AI model and version that made the identification, its confidence, the station's listening effort and a human review status.

> **Draft (October 2026).** The dataset is in onboarding with the AWS Open Data Sponsorship Program and the bucket is not public yet. Every cell below works today: set `USE_S3 = True` once the bucket is live; until then the same data is read from the WildNetwork public API, which serves identical records.

Full documentation and data dictionary: [README.md](README.md)

### Q: How have you organized your dataset? Help us understand the key prefix structure of your S3 bucket.

The bucket `wildnetwork-open-data` (us-east-1) has four kinds of content:

- `events/year=YYYY/month=MM/day=DD/part-NNN.parquet`: one row per detection event, as GeoParquet.
- `aggregates/species_daily/year=YYYY/` and `aggregates/species_weekly/year=YYYY/`: counts per species, 5-degree grid cell and day or ISO week. These are what movement and timing analyses need, and they are small.
- `schema/`, `manifest.json`, `README.md`, `LICENSE.md`, `CHANGELOG.md`: the WDX JSON Schema, the latest snapshot's file list and row counts, and documentation.
- `noncommercial/`: records whose observers chose CC BY-NC, with the same layout and columns. Everything outside this prefix is CC0 or CC BY, so leave it out of your paths if you need data you can use commercially.

Partitions use Hive-style `key=value` folders, so DuckDB, Polars, PyArrow, Spark and Amazon Athena prune by date automatically.

In [ ]:
# Libraries used in this notebook (requirements.txt format):
# boto3>=1.34
# duckdb>=1.1
# polars>=1.0
# requests>=2.31
# matplotlib>=3.8

import json
import boto3, duckdb, polars as pl, requests
import matplotlib.pyplot as plt
from botocore import UNSIGNED
from botocore.config import Config

USE_S3 = False                      # set to True once s3://wildnetwork-open-data/ is public
BUCKET = "wildnetwork-open-data"
REGION = "us-east-1"
API = "https://wildnetwork.arunrajiah.com/api/v1"

First, list the top level of the bucket. Public buckets need no credentials, so requests are unsigned.

In [ ]:
if USE_S3:
    s3 = boto3.client("s3", region_name=REGION, config=Config(signature_version=UNSIGNED))
    for p in s3.list_objects_v2(Bucket=BUCKET, Delimiter="/").get("CommonPrefixes", []):
        print(p["Prefix"])
    manifest = json.load(s3.get_object(Bucket=BUCKET, Key="manifest.json")["Body"])
    print(json.dumps(manifest, indent=2)[:1500])
else:
    # Until launch: the live totals the snapshot is built from
    status = requests.get(f"{API}/status", timeout=30).json()
    print({k: status[k] for k in ("sensors", "detections", "species")})

### Q: What file formats are used, and why?

- **Apache Parquet** for events and aggregates: columnar, compressed and readable directly from S3 by DuckDB, Polars, Spark and Athena without downloading whole files. Event files are **GeoParquet**, so the `geometry` column is understood by GeoPandas, DuckDB Spatial and QGIS.
- **JSON Schema** for the WDX format itself (`schema/wdx-0.2.schema.json`, also archived on Zenodo, DOI 10.5281/zenodo.22200864). The schema is the contract: every event in the bucket validated against it on the way in.

Three dataset-specific points matter more than the format:

1. `confidence` is a classifier score from 0 to 1 and **is not a probability**. Scores are not calibrated across species, regions or models.
2. `review_status` says whether a person checked the record (`confirmed`, `rejected`, `uncertain`) or not (`unreviewed`). Most records today are unreviewed machine output.
3. Locations are **generalised**: private stations to about 1 km, sensitive species further. Use `coordinate_uncertainty_m`.

### Q: Can you show us an example of downloading and loading data from your dataset?

Here we load recent detection events for southern India (Kerala, Tamil Nadu and Karnataka) and look at their structure.

In [ ]:
BBOX = (76.0, 8.0, 81.0, 14.0)   # west, south, east, north

if USE_S3:
    con = duckdb.connect()
    con.sql("INSTALL httpfs; LOAD httpfs; INSTALL spatial; LOAD spatial;")
    con.sql(f"SET s3_region='{REGION}'")
    events = con.sql(f'''
        SELECT event_id, event_start, scientific_name, vernacular_name, taxon_group,
               confidence, classifier_name, classifier_version, review_status, source_system,
               ST_X(ST_GeomFromWKB(geometry)) AS lon, ST_Y(ST_GeomFromWKB(geometry)) AS lat
        FROM read_parquet('s3://{BUCKET}/events/*/*/*/*.parquet', hive_partitioning=true)
        WHERE year = 2026 AND month = 10
          AND ST_X(ST_GeomFromWKB(geometry)) BETWEEN {BBOX[0]} AND {BBOX[2]}
          AND ST_Y(ST_GeomFromWKB(geometry)) BETWEEN {BBOX[1]} AND {BBOX[3]}
    ''').pl()
else:
    fc = requests.get(f"{API}/events", params={"bbox": ",".join(map(str, BBOX)), "limit": 2000}, timeout=60).json()
    events = pl.DataFrame([
        {**f["properties"], "lon": f["geometry"]["coordinates"][0], "lat": f["geometry"]["coordinates"][1]}
        for f in fc["features"]
    ]).rename({"sci": "scientific_name", "common": "vernacular_name", "conf": "confidence",
               "source": "source_system", "t": "event_start", "group": "taxon_group"})

print(events.shape)
events.head()

Which species dominate the most recent detections, and how confident is the classifier about them?

In [ ]:
(events.group_by("scientific_name", "vernacular_name")
       .agg(pl.len().alias("detections"),
            pl.col("confidence").median().alias("median_confidence"))
       .sort("detections", descending=True)
       .head(15))

### Q: A picture is worth a thousand words. Show us a visual from your dataset.

The weekly aggregates make migration visible. Below, the Barn Swallow's detections per 5-degree cell are averaged into a latitude for every week of the past year, separately for its African wintering range and European breeding range. The northward spring push and the autumn return show up without any modelling.

In [ ]:
SPECIES = "Hirundo rustica"   # Barn Swallow

if USE_S3:
    weekly = con.sql(f'''
        SELECT week, cell_lat + 2.5 AS lat, cell_lon + 2.5 AS lon, sum(count) AS n
        FROM read_parquet('s3://{BUCKET}/aggregates/species_weekly/*/*.parquet', hive_partitioning=true)
        WHERE scientific_name = '{SPECIES}'
        GROUP BY ALL
    ''').pl()
else:
    frames = requests.get(f"{API}/species/{SPECIES}/movement", timeout=60).json()["frames"]
    weekly = pl.DataFrame([
        {"week": fr["week"], "lat": c[0] + 2.5, "lon": c[1] + 2.5, "n": c[2]}
        for fr in frames for c in fr["cells"]
    ]).with_columns(pl.col("week").str.to_date())

# Split Old World cells into Africa (south of 15 N) and Europe (north of 35 N), weighted mean latitude per week
def mean_lat(df, cond):
    return (df.filter(cond).group_by("week")
              .agg((pl.col("lat") * pl.col("n")).sum() / pl.col("n").sum())
              .rename({"lat": "mean_lat"}).sort("week"))

old_world = weekly.filter((pl.col("lon") > -20) & (pl.col("lon") < 60))
africa = mean_lat(old_world, pl.col("lat") < 15)
europe = mean_lat(old_world, pl.col("lat") > 35)

fig, ax = plt.subplots(figsize=(11, 5), dpi=100)
ax.plot(africa["week"], africa["mean_lat"], color="#c8833a", lw=2, label="Africa (wintering)")
ax.plot(europe["week"], europe["mean_lat"], color="#2f6b4f", lw=2, label="Europe (breeding)")
ax.set_title("Barn Swallow: detection-weighted latitude by week", fontsize=14, weight="bold")
ax.set_ylabel("Mean latitude of detections (degrees)")
ax.grid(alpha=0.3); ax.legend(frameon=False)
plt.tight_layout(); plt.show()

### Q: What is one example of a question you can answer with these data?

**When do migrants arrive in each region, and is that changing?** WildNetwork publishes, for each species and 5-degree cell, an arrival, peak and departure week, with an uncertainty interval and a support score (the share of resampled estimates that agree). The method uses each species' share of all detections in the cell, so more stations do not look like earlier arrivals. Here we load arrival weeks for the Barn Swallow and keep only well-supported estimates.

In [ ]:
import io
if USE_S3:
    arrivals = None   # published as aggregates/arrivals/ after launch; until then use the API
csv = requests.get(f"{API}/arrivals", params={"species": SPECIES, "format": "csv"}, timeout=60).text
arrivals = pl.read_csv(io.StringIO(csv), comment_prefix="#")
good = (arrivals.filter(pl.col("arrivalSupport") >= 0.5)
                .select("region", "cellLat", "cellLon", "arrivalWeek", "arrivalLow", "arrivalHigh", "peakWeek", "detections")
                .sort("cellLat"))
good

In Europe, from 45 degrees north upwards, the median arrival week moves later with latitude: late April at 45 to 55 degrees north, mid-May at 60 degrees, June at 65 degrees. That is the expected pattern of spring passage, recovered from community detections alone. With a few more years of snapshots, the same table per year becomes a direct measure of phenology shift, cell by cell, for every species with enough detections. Methods and their validation against independent records: <https://wildnetwork.arunrajiah.com/methods> and <https://wildnetwork.arunrajiah.com/validation>.

### Q: What is one unanswered question that you think could be answered using this dataset?

**Can we calibrate machine detections for under-sampled regions, so that unreviewed records become usable?**

Classifiers such as BirdNET and Perch perform worse in Africa and Asia than in Europe and North America, and their confidence scores mean different things for different species. Most records in this dataset are unreviewed. The challenge: using the growing set of `confirmed` and `rejected` records (starting with WildNetwork's planned review pilot in a Chennai wetland), estimate per-species, per-region precision as a function of confidence, and publish thresholds or call-density estimates that let researchers use the unreviewed bulk of the data with known error rates. A good answer would turn hundreds of millions of raw detections into calibrated evidence of where species are, especially in the regions with the least monitoring today.